In [ ]:
# CELL 1: IMPORTS
# ============================================================================
import sys
import os
from pathlib import Path

# Add project root to Python path
# When running from notebooks/, we need to go up one level to get to project root
notebook_path = Path().resolve()
if notebook_path.name == 'notebooks':
    project_root = notebook_path.parent
else:
    project_root = notebook_path  # Already in project root

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy import stats
import warnings
import importlib
warnings.filterwarnings('ignore')

# Import from our modules
# Force reload to pick up any changes to the module
import quagua.data_utils
importlib.reload(quagua.data_utils)
from quagua.data_utils import load_titanic_data
from quagua.encoders import (
    # Irreversible encoders (for encoding stage)
    # RandomOneToManyEncoder,  # COMMENTED: not compatible with continuous data
    ChaoticSystemEncoder,  # ✅ Works with continuous
    OrthogonalSubspaceEncoder,  # ✅ Works with continuous
    FastPrivacyEncoder,  # ✅ NEW: Fast privacy encoder (like FHE but faster)
    # Mixers (for mixing stage)
    ClassicalXORMixer,  # ✅ Works with continuous
    QuantumEntanglementEncoder,  # ✅ Works with continuous
    QuantumWalkEncoder,  # ✅ Works with continuous
    QuantumSuperpositionEncoder,  # ✅ Works with continuous
    QuantumPhaseEncoder,  # ✅ Works with continuous
    QuantumFingerprintingEncoder,  # ✅ Quantum fingerprinting
    # Pipeline
    EncodeThenMixPipeline,
)
from quagua.evaluation import PrivacyAITestFramework

# Set random seeds for reproducibility
np.random.seed(42)

# ============================================================================
# CELL 2: DATA LOADING
# ============================================================================
print("🚀 ENCODING + MIXING PIPELINE EVALUATION")
print("="*60)

print("\n📊 Loading Titanic dataset...")
# Include continuous numerical features (Age, Fare) along with categorical/binary
X, y, feature_names = load_titanic_data(include_continuous=True)
print(f"Loaded {X.shape[0]} samples with {X.shape[1]} features")
print(f"Features: {feature_names}")
print(f"Class distribution: Survived={y.mean():.2f}, Not Survived={1-y.mean():.2f}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

# ============================================================================
# CELL 3: DEFINE ENCODING + MIXING PIPELINES
# ============================================================================
# Encoder classes live in quagua.encoders modules
# Focus on encoding+mixing combinations only

# Define encoding+mixing pipelines
encoders = {
    # Baseline for comparison
    'Baseline (No Encoding)': lambda x: x,
    
    # ========================================================================
    # ENCODING + MIXING PIPELINES
    # ========================================================================
    # NOTE: RandomOneToManyEncoder methods are commented out because they
    # don't work well with continuous numerical data (too many unique values)
    
    # # 1-to-Many Encoding + Various Mixers (COMMENTED: not compatible with continuous)
    # '1-to-Many + XOR Mix': EncodeThenMixPipeline(
    #     RandomOneToManyEncoder(n_encodings_per_value=5, encoding_dim=3),
    #     ClassicalXORMixer()
    # ),
    # '1-to-Many + Entanglement': EncodeThenMixPipeline(
    #     RandomOneToManyEncoder(n_encodings_per_value=5, encoding_dim=3),
    #     QuantumEntanglementEncoder(entanglement_strength=0.7)
    # ),
    # '1-to-Many + Quantum Walk': EncodeThenMixPipeline(
    #     RandomOneToManyEncoder(n_encodings_per_value=5, encoding_dim=3),
    #     QuantumWalkEncoder(n_steps=3, graph_type='complete')
    # ),
    
    # Chaotic Encoding + Various Mixers (✅ Compatible with continuous data)
    # IMPROVED: Reduced iterations (10->5) to preserve more information
    'Chaotic (Logistic) + XOR Mix': EncodeThenMixPipeline(
        ChaoticSystemEncoder(map_type='logistic', n_iterations=5),
        ClassicalXORMixer()
    ),
    # IMPROVED: Reduced iterations (10->5) and entanglement (0.7->0.4) for better balance
    'Chaotic (Logistic) + Entanglement': EncodeThenMixPipeline(
        ChaoticSystemEncoder(map_type='logistic', n_iterations=5),
        QuantumEntanglementEncoder(entanglement_strength=0.4)
    ),
    # IMPROVED: Reduced iterations (10->5) to preserve accuracy
    'Chaotic (Logistic) + Quantum Walk': EncodeThenMixPipeline(
        ChaoticSystemEncoder(map_type='logistic', n_iterations=5),
        QuantumWalkEncoder(n_steps=3, graph_type='complete')
    ),
    # IMPROVED: Reduced iterations (10->5) and entanglement (0.7->0.4)
    'Chaotic (Henon) + Entanglement': EncodeThenMixPipeline(
        ChaoticSystemEncoder(map_type='henon', n_iterations=5),
        QuantumEntanglementEncoder(entanglement_strength=0.4)
    ),
    # NEW: Gentler chaotic encoding with fewer iterations
    'Chaotic (Logistic, gentle) + Quantum Walk': EncodeThenMixPipeline(
        ChaoticSystemEncoder(map_type='logistic', n_iterations=3),
        QuantumWalkEncoder(n_steps=2, graph_type='complete')
    ),
    
    # Orthogonal Subspace Encoding + Various Mixers (✅ Compatible with continuous data)
    'Orthogonal + XOR Mix': EncodeThenMixPipeline(
        OrthogonalSubspaceEncoder(subspace_dim=3),
        ClassicalXORMixer()
    ),
    # IMPROVED: Reduced entanglement (0.7->0.4) for better accuracy/privacy balance
    'Orthogonal + Entanglement': EncodeThenMixPipeline(
        OrthogonalSubspaceEncoder(subspace_dim=3),
        QuantumEntanglementEncoder(entanglement_strength=0.4)
    ),
    'Orthogonal + Quantum Walk': EncodeThenMixPipeline(
        OrthogonalSubspaceEncoder(subspace_dim=3),
        QuantumWalkEncoder(n_steps=3, graph_type='complete')
    ),
    # NEW: Gentler orthogonal encoding with reduced mixing
    'Orthogonal (gentle) + Quantum Walk': EncodeThenMixPipeline(
        OrthogonalSubspaceEncoder(subspace_dim=2),  # Smaller subspace
        QuantumWalkEncoder(n_steps=2, graph_type='complete')  # Fewer steps
    ),
    
    # Advanced combinations (✅ Compatible with continuous data)
    # IMPROVED: Reduced iterations (10->5) for better accuracy
    'Chaotic (Henon) + Quantum Phase': EncodeThenMixPipeline(
        ChaoticSystemEncoder(map_type='henon', n_iterations=5),
        QuantumPhaseEncoder(n_phases=3, add_noise=True, noise_level=0.02)
    ),
    # NEW: Very gentle encoding for maximum accuracy preservation
    'Chaotic (Logistic, minimal) + Entanglement': EncodeThenMixPipeline(
        ChaoticSystemEncoder(map_type='logistic', n_iterations=2),
        QuantumEntanglementEncoder(entanglement_strength=0.3)
    ),
    # '1-to-Many + Quantum Superposition': EncodeThenMixPipeline(
    #     RandomOneToManyEncoder(n_encodings_per_value=5, encoding_dim=3),
    #     QuantumSuperpositionEncoder(n_states=3, temperature=0.5)
    # ),  # COMMENTED: not compatible with continuous
    
    # ========================================================================
    # QUANTUM FINGERPRINTING (NEW)
    # ========================================================================
    # Quantum fingerprinting: Creates compact fingerprint using quantum measurements
    'Quantum Fingerprinting (default)': QuantumFingerprintingEncoder(
        fingerprint_dim=None,  # Same as input dimension
        n_measurements=3,
        interference_strength=0.5
    ),
    'Quantum Fingerprinting (expanded)': QuantumFingerprintingEncoder(
        fingerprint_dim=14,  # 2x expansion
        n_measurements=3,
        interference_strength=0.5
    ),
    'Quantum Fingerprinting (strong interference)': QuantumFingerprintingEncoder(
        fingerprint_dim=None,
        n_measurements=4,
        interference_strength=0.7  # Stronger quantum interference
    ),
    
    # ========================================================================
    # FAST PRIVACY ENCODER (NEW - Like FHE but faster)
    # ========================================================================
    # Fast, one-way encoding optimized for binary/categorical data
    # Works like fully homomorphic encryption but much faster
    # No decoding needed (irreversible), preserves ML utility
    'Fast Privacy (default)': FastPrivacyEncoder(
        encoding_dim=4,          # 4 dimensions per feature
        polynomial_degree=1,     # Linear expansion (fast, less expansion)
        noise_level=0.1,         # Low noise (preserves utility)
        use_mixing=True,         # Feature mixing enabled
        hash_seed=42             # Reproducible
    ),
    'Fast Privacy (high noise)': FastPrivacyEncoder(
        encoding_dim=4,
        polynomial_degree=1,
        noise_level=0.3,         # Higher noise (more privacy, less utility)
        use_mixing=True,
        hash_seed=42
    ),
    'Fast Privacy (expanded)': FastPrivacyEncoder(
        encoding_dim=6,          # More dimensions per feature
        polynomial_degree=1,
        noise_level=0.1,
        use_mixing=True,
        hash_seed=42
    ),
    'Fast Privacy (polynomial)': FastPrivacyEncoder(
        encoding_dim=4,
        polynomial_degree=2,     # Quadratic expansion (more utility, more expansion)
        noise_level=0.1,
        use_mixing=True,
        hash_seed=42
    ),
}

print(f"\n📋 Defined {len(encoders)} encoding methods to test")
print(f"   - {len([k for k in encoders.keys() if '+' in k])} encoding+mixing pipelines")
print(f"   - 1 baseline (no encoding)")

# ============================================================================
# CELL 4: RUN EVALUATION
# ============================================================================
# Framework class is in quagua.evaluation module
framework = PrivacyAITestFramework()

print("\n" + "="*60)
print("RUNNING COMPREHENSIVE EVALUATION")
print("="*60)

all_results = framework.run_comparison(X_train, X_test, y_train, y_test, encoders=encoders)


In [ ]:
# ============================================================================
# CELL 5: VISUALIZE RESULTS
# ============================================================================
framework.plot_results(all_results)


In [ ]:
# ============================================================================
# CELL 6: STATISTICAL TESTING (OPTIONAL)
# ============================================================================
# Select a subset of pipelines for statistical testing (faster)
stat_encoders = {
    'Baseline (No Encoding)': lambda x: x,
    'Chaotic (Logistic) + Entanglement': encoders['Chaotic (Logistic) + Entanglement'],
    'Orthogonal + Quantum Walk': encoders['Orthogonal + Quantum Walk'],
}

print("\n" + "="*60)
print("STATISTICAL SIGNIFICANCE TESTING")
print("="*60)
stat_results = framework.compare_methods_statistically(
    X_train, X_test, y_train, y_test, 
    encoders=stat_encoders, 
    n_runs=3
)
